# **🚀 Факторизация и другие интересные операции в Julia**  
# **Урок Тринадцатый.**

По мотивам работы Андреаса Ноака Йенсена (MIT & JuliaComputing) [(Twitter)](https://x.com/anoackjensen)  
(с авторскими дополнениями Сергея Соболевского)

<br>

##  **📌 Темы:**  

1. **Что такое факторизация матрицы**
   
2. **Основные виды факторизации матриц.**
   
   2.1 **LU-факторизация**

   2.2 **QR-факторизация**

   2.3 **Cholesky-факторизация**

   2.4 **Сингулярное разложение (SVD)**

   2.5 **Разложение по собственным значениям и собственным векторам (EVD)**

   
3. **Специальные матричные структуры.**
   
4. **Общая линейная алгебра.**



<br>


<br>


## 🎯 После этого занятия вы сможете

- объяснять, **зачем** нужны факторизации, а не только вызывать функции;
- выбирать разложение под задачу: LU, QR, Cholesky, SVD, спектральное;
- проверять **предусловия** разложения (`issymmetric`, `isposdef`) и понимать,
  что произойдёт при их нарушении;
- **переиспользовать** одну факторизацию для многих правых частей;
- пользоваться типами структуры (`Symmetric`, `Diagonal`, `SymTridiagonal`),
  чтобы Julia выбирала специализированный алгоритм;
- работать с обобщённой линейной алгеброй (рациональные числа) там,
  где важна точность.

### Связь с предыдущими занятиями

В **занятии 11** мы разобрали смысл операций, в **занятии 12** — библиотеку
`LinearAlgebra` и оператор `\`. Здесь мы открываем «чёрный ящик»: смотрим,
что именно `\` делает внутри, и учимся управлять этим выбором.

<br>

---

<br>

# 📚 **1. Что такое факторизация матрицы?**

**Факторизация матрицы** (или разложение матрицы) — это представление исходной матрицы в виде произведения нескольких матриц определённого типа, обычно более простых по структуре. 

Суть метода — исходную задачу (например, решение системы линейных уравнений, нахождение определителя или обратной матрицы) привести к более простой и численно устойчивой форме.

<br>


### 🧮 Для чего нужна факторизация?

Факторизация применяется в различных областях математики и вычислительной науки:

- **Решение систем линейных уравнений**: упрощает решение, особенно при множестве задач с одинаковой матрицей, но разными правыми частями.
- **Нахождение обратных матриц**: облегчает и ускоряет процесс вычисления.
- **Определение ранга матрицы и её структуры** (например, собственных значений и векторов).
- **Упрощение вычислений** в численных методах, повышая устойчивость и точность расчётов.

<br>



<br>

<br>


### ⚙️ Настройка нашей среды **Julia**.

Прежде чем начать, давайте настроим нашу среду , а заодно повторим как подключить в **Julia** библиотеку `LinearAlgebra`, чтобы дальше мы могли применять факторизации и работать со специальными структурами матриц.  

- **Подключаем пакет `LinearAlgebra` :**

In [1]:
using LinearAlgebra # Подключаем пакет LinearAlgebra



- **Создаем случайную матрицу $A$ (3x3) :**

In [2]:
A = rand(3, 3)  # Создаем случайную матрицу 3x3

3×3 Matrix{Float64}:
 0.311646  0.0732776  0.44705
 0.23844   0.877903   0.472878
 0.936196  0.687625   0.145605

- **Создаем вектор $x$ из трех единиц :**

In [3]:
x = fill(1, (3,))  # Создаем вектор из трех 1-единиц

3-element Vector{Int64}:
 1
 1
 1

- **Вычисляем для пробы результат умножения матрицы $A$ на вектор $x$ :**

In [4]:
b = A * x  # Вычисляем результат умножения матрицы A на вектор x

3-element Vector{Float64}:
 0.8319742305031733
 1.589221476266962
 1.769425577341556

**🚩 "Бинго!" Всё получилось, идём дальше!**

<br>

---

<br>

## **🔢 2. Основные виды факторизации матриц.**

<br>
Наиболее часто применяются следующие виды разложений:

1. **LU-факторизация**
   
2. **QR-факторизация**
   
3. **Cholesky-факторизация**
   
4. **Сингулярное разложение (SVD)**
   
5. **Разложение по собственным значениям и собственным векторам (EVD)**

<br>

Давайте рассмотрим кратко каждый этот тип подробнее, но вначале настроим нашу среду **Julia** для работы.

<br>

---

<br>

### 🟢 **2.1 LU-факторизация**

<br>

**LU-факторизация** — это представление матрицы $A$ в виде произведения нижней треугольной матрицы $L$ (Lower triangular) и верхней треугольной матрицы $U$ (Upper triangular):

$$
A = L \cdot U
$$

Чтобы повысить численную устойчивость вычислений, обычно используется матрица перестановок $P$:

$$
P \cdot A = L \cdot U
$$

Здесь:

- $P$ — матрица перестановок (переставляет строки исходной матрицы $A$),
- $L$ — нижняя треугольная матрица с единичными элементами на главной диагонали,
- $U$ — верхняя треугольная матрица.

<br>

#### 🔹 **Применение LU-факторизации:**

- Решение систем линейных уравнений.
- Вычисление определителей матриц.
- Нахождение обратной матрицы.


В языке Julia LU-факторизация матрицы $A$ выполняется с помощью функции `lu()`:

In [5]:
Alu = lu(A)  # Разложение матрицы A на L и U

LU{Float64, Matrix{Float64}, Vector{Int64}}
L factor:
3×3 Matrix{Float64}:
 1.0        0.0       0.0
 0.254691   1.0       0.0
 0.332886  -0.221442  1.0
U factor:
3×3 Matrix{Float64}:
 0.936196  0.687625  0.145605
 0.0       0.702772  0.435793
 0.0       0.0       0.495083

👉 Julia автоматически создаёт объект специального составного типа, который хранит компоненты факторизации.

Извлечь компоненты факторизации можно с помощью специальных полей объекта:

- **Матрица перестановок:**

In [6]:
P = Alu.P

3×3 Matrix{Float64}:
 0.0  0.0  1.0
 0.0  1.0  0.0
 1.0  0.0  0.0

- **Нижняя треугольная матрица $L$ :**

In [7]:
L = Alu.L

3×3 Matrix{Float64}:
 1.0        0.0       0.0
 0.254691   1.0       0.0
 0.332886  -0.221442  1.0

- **Верхняя треугольная матрица $U$ :**

In [8]:
U = Alu.U

3×3 Matrix{Float64}:
 0.936196  0.687625  0.145605
 0.0       0.702772  0.435793
 0.0       0.0       0.495083

### 🗒 **Примеры использования LU-факторизации:**

#### 🔹 **Пример 1: Решение системы линейных уравнений:**

Пусть задана система линейных уравнений:

$$
\begin{cases}
2x + 3y = 5 \\
4x + 5y = 11
\end{cases}
$$

Запишем в матричной форме:

In [9]:
A = [2 3; 4 5]
b = [5; 11]

2-element Vector{Int64}:
  5
 11

In [10]:
Alu = lu(A) # Разложение матрицы A на L и U

LU{Float64, Matrix{Float64}, Vector{Int64}}
L factor:
2×2 Matrix{Float64}:
 1.0  0.0
 0.5  1.0
U factor:
2×2 Matrix{Float64}:
 4.0  5.0
 0.0  0.5

- **Решение через исходную матрицу:**

In [11]:
x_direct = A \ b  # Решение системы уравнений Ax = b с помощью обратного хода

2-element Vector{Float64}:
  4.0
 -1.0

- **Решение через факторизацию LU:**

In [12]:
x_lu = Alu \ b  # Решение системы уравнений Ax = b с помощью LU-разложения

2-element Vector{Float64}:
  4.0
 -1.0

**Убеждаемся, что оба метода дают одинаковый результат.**


#### 🔹 **Пример 2: Вычисление определителя:**

Определитель матрицы можно вычислить напрямую или используя LU-факторизацию:

In [13]:
det_A_direct = det(A) # Вычисление определителя матрицы A с помощью обратного хода

-2.0

In [14]:

det_A_lu = det(Alu) # Вычисление определителя матрицы A с помощью LU-разложения

-2.0

Результаты будут близки до численной погрешности:

In [15]:
det(A) ≈ det(Alu) # Проверка равенства определителей

true

#### 👉 **Преимущества LU-факторизации в Julia:**

- **Численная стабильность**: Благодаря матрице перестановок факторизация становится устойчивой.
  
- **Эффективность**: LU-факторизацию можно использовать повторно для различных вычислений (решение систем, нахождение обратной матрицы, вычисление определителей).



<br>

---

<br>


<br>

---

<br>

### 🔁 Главная причина существования факторизаций: переиспользование

Вот тот случай, ради которого всё и затевалось.

Пусть нужно решить систему с **одной и той же** матрицей, но многими разными
правыми частями: $Ax_1 = b_1$, $Ax_2 = b_2$, …, $Ax_k = b_k$.
Такое встречается постоянно — шаги по времени, серии измерений, метод Ньютона.

Если каждый раз писать `A \ bᵢ`, Julia **каждый раз заново** раскладывает `A`.
Разложение стоит $O(n^3)$, а решение с готовым разложением — всего $O(n^2)$.
Значит, разложив матрицу один раз, все последующие решения мы получаем почти даром.

In [16]:
using LinearAlgebra, BenchmarkTools

n  = 300
Ar = randn(n, n) + n * I          # хорошо обусловленная матрица
bs = [randn(n) for _ in 1:50]     # 50 разных правых частей

50-element Vector{Vector{Float64}}:
 [1.4126533704261257, 0.5779484697276649, 0.10784862469781907, 0.34867608842989006, 0.5905822481595296, -0.7670588758543648, 0.07444346498984639, 0.9136545719238789, 1.918742462531711, -1.2061893225175966  …  1.4556400306428294, 0.6313180052161692, 0.5979312446645948, -0.6534109227473912, 0.6652873089931962, 0.984366563599073, -0.7226870256675322, 0.8703652643176311, 0.36385017367777944, 0.3868979484886434]
 [-0.9645811991522943, 0.9807826985561792, -1.070900679159436, -1.2839954951906627, -0.9467437082661803, -0.32949375879396936, 0.3389095238213785, -0.00033705700614926446, -0.27994484538734976, -0.43405824567097284  …  -0.16024422508261787, -1.8192487964281758, 1.3177348934028303, -1.0184918533211111, 0.2816606635939608, -1.3516671805945473, -1.0695636575993255, 0.5873238728106, 0.03141017571577973, 0.7474408702523595]
 [0.6177390677138003, -0.09497639970547214, -1.3564149917138468, -0.9072526093467345, -0.02592315607756526, -1.576598126996329, 1.

In [17]:
# Вариант 1: наивно — каждый раз заново раскладываем матрицу
naive(A, rhs) = [A \ b for b in rhs]

# Вариант 2: раскладываем один раз, дальше только подстановки
F = lu(Ar)
reused(fact, rhs) = [fact \ b for b in rhs]

@show naive(Ar, bs) ≈ reused(F, bs)      # результат тот же

naive(Ar, bs) ≈ reused(F, bs) = true

true

In [18]:
# Аргументы подставляем через $, а сами данные передаём в функцию —
# так измеряется работа, а не обращение к глобальным переменным
# (подробнее об этом — в занятии 10).
t_naive  = @belapsed naive($Ar, $bs)
t_reused = @belapsed reused($F, $bs)

println("каждый раз заново : ", round(t_naive  * 1000, digits = 1), " мс")
println("одно разложение   : ", round(t_reused * 1000, digits = 1), " мс")
println("ускорение         : ", round(t_naive / t_reused, digits = 1), "×")

каждый раз заново : 205.3

 мс
одно разложение   : 2.5 мс
ускорение         : 81.3×


Абсолютные числа зависят от машины, но соотношение устойчиво: чем больше правых
частей, тем заметнее выигрыш.

> 📌 **Объект факторизации ведёт себя как матрица.** `F \ b` работает, `det(F)`
> работает, но внутри уже лежит готовое разложение. Именно поэтому в занятии 12
> мы говорили, что `\` «сам выбирает алгоритм»: он раскладывает матрицу, решает
> систему и **выбрасывает** разложение. Если оно нужно снова — сохраните его сами.

Тот же приём работает и для других разложений — `cholesky`, `qr`:

In [19]:
S  = Ar'Ar + I                 # симметричная положительно определённая
Fc = cholesky(S)               # разложение считаем один раз

x1 = Fc \ bs[1]
x2 = Fc \ bs[2]                # второе решение — почти бесплатно

@show norm(S * x1 - bs[1])
@show norm(S * x2 - bs[2])

norm(S * x1 - bs[1]) = 6.56217654669121e-15


norm(S * x2 - bs[2]) = 6.918728858045285e-15


6.918728858045285e-15

<br>

### 🟢 **2.2 QR-факторизация**

**QR-факторизация** — это разложение матрицы $ A $ в виде произведения двух специальных матриц: ортогональной (унитарной) матрицы $ Q $ и верхней треугольной матрицы $ R $ :

<br>

$$
A = Q \cdot R
$$

**Характеристики матриц разложения:**

- Матрица $ Q $ — ортогональная (унитарная), то есть выполняется:
$$
Q^T Q = I
$$

Это означает, что столбцы матрицы $ Q $ :

- **ортогональны** друг другу (их скалярное произведение равно нулю);
- имеют **единичную длину** (нормированы).

- Матрица $ R $ — верхняя треугольная, то есть все её элементы ниже главной диагонали равны нулю.


<br>

#### 📌 **Применение QR-факторизации:**

QR-факторизация широко используется в вычислительной математике, статистике и инженерии:

- **Решение задач наименьших квадратов** (линейная регрессия).
  
- **Решение линейных систем уравнений** (особенно устойчивых к численным погрешностям).
  
- **Вычисление собственных значений и векторов** (QR-алгоритм).

<br>

### 🗒  **Примеры выполнения QR-факторизации в Julia:**

#### 🔹 **Пример 1.**

В языке Julia **QR-факторизацию** матрицы $ A $ можно выполнить с помощью функции `qr()`:


In [20]:
Aqr = qr(A) # Разложение матрицы A на Q и R

LinearAlgebra.QRCompactWY{Float64, Matrix{Float64}, Matrix{Float64}}
Q factor: 2×2 LinearAlgebra.QRCompactWYQ{Float64, Matrix{Float64}, Matrix{Float64}}
R factor:
2×2 Matrix{Float64}:
 -4.47214  -5.81378
  0.0      -0.447214


После этого результат факторизации сохраняется в специальном объекте, из которого легко извлечь матрицы $ Q $ и $ R $ :

- **Матрица $ Q $** (ортогональная матрица):

In [21]:
Q = Aqr.Q

2×2 LinearAlgebra.QRCompactWYQ{Float64, Matrix{Float64}, Matrix{Float64}}

- **Матрица $ R $** (верхняя треугольная матрица):

In [22]:
R = Aqr.R

2×2 Matrix{Float64}:
 -4.47214  -5.81378
  0.0      -0.447214

👉 Таким образом, после выполнения **QR-факторизации** мы можем работать с её компонентами напрямую и эффективно.

#### 🔹 **Пример 2.**

**Краткий пример использования, :) для закрепления темы:**

Рассмотрим простую матрицу:

```julia
A = [1 2; 
     3 4; 
     5 6]
```

In [23]:
A = [1 2; 3 4; 5 6]

3×2 Matrix{Int64}:
 1  2
 3  4
 5  6

In [24]:
Aqr = qr(A) # Разложение матрицы A на Q и R

LinearAlgebra.QRCompactWY{Float64, Matrix{Float64}, Matrix{Float64}}
Q factor: 3×3 LinearAlgebra.QRCompactWYQ{Float64, Matrix{Float64}, Matrix{Float64}}
R factor:
2×2 Matrix{Float64}:
 -5.91608  -7.43736
  0.0       0.828079

In [25]:
Q = Aqr.Q   # ортогональная матрица

3×3 LinearAlgebra.QRCompactWYQ{Float64, Matrix{Float64}, Matrix{Float64}}

In [26]:
R = Aqr.R   # верхняя треугольная матрица

2×2 Matrix{Float64}:
 -5.91608  -7.43736
  0.0       0.828079

👉 **Теперь исходная матрица $ A $ может быть представлена в виде:**

$$
A = Q \cdot R
$$


In [27]:
A = Q * R  # Проверка разложения

3×2 Matrix{Float64}:
 1.0  2.0
 3.0  4.0
 5.0  6.0

<br>

#### 👉 **Преимущества использования QR-факторизации:**

- **Численная устойчивость**: QR-факторизация обычно более стабильна и точна, чем другие методы (например, LU-факторизация), особенно при работе с плохо обусловленными матрицами.
  
- **Эффективность**: Позволяет быстро и устойчиво решать системы линейных уравнений и задачи регрессии.



<br>

---

<br>

<br>

### 🟢 **2.3 Cholesky-факторизация**

**Cholesky-факторизация** применяется исключительно для **симметричных и положительно определённых** матриц. Она представляется в виде произведения нижней треугольной матрицы $L$ и её транспонированной матрицы $L^T$:

$$
A = L \cdot L^T
$$

где:

- $L$ — нижняя треугольная матрица (все элементы выше диагонали равны нулю).
- $L^T$ — верхняя треугольная матрица, транспонированная к $L$.

<br>

##### В Julia **Cholesky-факторизация** выполняется функцией `cholesky()` .

<br>

### 🔹 **Особенности Cholesky-факторизации:**

- Используется только для симметричных и положительно определённых матриц.
- Является более простым и численно стабильным методом по сравнению с LU-факторизацией.
- Более эффективна с точки зрения производительности, так как требует примерно в два раза меньше вычислений, чем LU-факторизация.

<br>


#### 📌 **Применение Cholesky-факторизации:**

- Решение систем линейных уравнений (особенно с симметричными матрицами).
  
- Вычисления в статистике (например, линейные регрессионные модели, многомерное нормальное распределение).
  
- Численные методы оптимизации и анализа данных.

<br>


### 🗒  **Примеры Cholesky-факторизации в Julia:**

#### 🔹 **Пример 1.**

1. **Определяем симметричную положительно определённую матрицу $A$:**

```julia
A = [
    25.0  15.0  -5.0;
    15.0  18.0   0.0;
    -5.0   0.0  11.0
]
```


In [28]:
A = [
    25.0  15.0  -5.0;
    15.0  18.0   0.0;
    -5.0   0.0  11.0
]

3×3 Matrix{Float64}:
 25.0  15.0  -5.0
 15.0  18.0   0.0
 -5.0   0.0  11.0

2. **Выполняем проверку симметричности :**

In [29]:
# Симметричность — только половина условия
println("issymmetric(A) = ", issymmetric(A))

# Вторая половина — положительная определённость.
# Именно её проверяет isposdef, и именно она нужна разложению Холецкого.
println("isposdef(A)    = ", isposdef(A))

# Эквивалентный, но более дорогой способ убедиться в том же:
println("собственные значения = ", eigvals(A))   # все строго положительны

issymmetric(A) = true


isposdef(A)    = 

true
собственные значения = 

[4.495462498263426, 12.015683645427234, 37.48885385630933]


3. **Выполнение Cholesky-факторизации :**

In [30]:
cholesky_factor = cholesky(A) # Разложение матрицы A на L и L^T

Cholesky{Float64, Matrix{Float64}}
U factor:
3×3 UpperTriangular{Float64, Matrix{Float64}}:
 5.0  3.0  -1.0
  ⋅   3.0   1.0
  ⋅    ⋅    3.0

4. **Выводим результат, т.е. извлекаем компоненты факторизации следующим образом:**

- **Нижняя треугольная матрица $L$ :**

In [31]:
L = cholesky_factor.L # нижняя треугольная матрица

3×3 LowerTriangular{Float64, Matrix{Float64}}:
  5.0   ⋅    ⋅ 
  3.0  3.0   ⋅ 
 -1.0  1.0  3.0

In [32]:
println("Cholesky factor L:\n", cholesky_factor.L) # Матрица L

Cholesky factor L:


[5.0 0.0 0.0; 3.0 3.0 0.0; -1.0 1.0 3.0]


### ⚠️ Что будет, если предусловие нарушено

Разложение Холецкого существует **только** для симметричных (эрмитовых)
положительно определённых матриц. Это не рекомендация, а условие существования:
на диагонали $L$ стоят квадратные корни, и при нарушении условия под корнем
оказывается отрицательное число.

Возьмём симметричную, но **не** положительно определённую матрицу:

In [33]:
B = [1.0  2.0
     2.0  1.0]

@show issymmetric(B)     # true — симметрична
@show eigvals(B)         # [-1.0, 3.0] — есть отрицательное собственное значение
@show isposdef(B)        # false — значит, разложения Холецкого не существует

issymmetric(B) = true
eigvals(B) = 

[-1.0, 3.0]
isposdef(B) = false


false

In [34]:
# Попытка всё равно разложить приведёт к ошибке. Это ожидаемое поведение:
cholesky(B)

LoadError: PosDefException: matrix is not positive definite; Factorization failed.

Выше — **намеренная ошибка**. `PosDefException` — это не поломка курса, а
корректная реакция Julia: матрица не удовлетворяет условию.

Если вы не уверены в матрице заранее, есть безопасный вариант — `cholesky`
с ключом `check=false`, который возвращает объект с полем `.info` вместо
исключения:

In [35]:
F = cholesky(B; check = false)

@show issuccess(F)      # false — разложение не удалось
@show F.info            # номер ведущего минора, на котором всё сломалось

# Идиоматичная проверка перед вычислением:
if isposdef(B)
    println("можно применять cholesky")
else
    println("матрица не положительно определена — берём LU или ldlt")
end

issuccess(F) = false
F.info = 2
матрица не положительно определена — берём LU или ldlt


> 📌 **Практическое правило.** `cholesky` примерно вдвое дешевле `lu`
> и требует вдвое меньше памяти — но только для СПО-матриц.
> Проверяйте `isposdef`, а не надейтесь на удачу.

#### 🔹 **Пример 2. Использования Cholesky-факторизации, ещё один:** :*

Рассмотрим симметричную и положительно определённую матрицу:

In [36]:
A = [4 2; 2 3]

2×2 Matrix{Int64}:
 4  2
 2  3

- **Проверяем положительную определённость (собственные значения положительны):**

In [37]:
eigvals(A) # Вычисление собственных значений матрицы A

2-element Vector{Float64}:
 1.43844718719117
 5.561552812808831


- **Выполняем Cholesky-факторизацию:**

In [38]:
A_chol = cholesky(A)

Cholesky{Float64, Matrix{Float64}}
U factor:
2×2 UpperTriangular{Float64, Matrix{Float64}}:
 2.0  1.0
  ⋅   1.41421

- **Извлекаем нижнюю треугольную матрицу L :**

In [39]:
L = A_chol.L

2×2 LowerTriangular{Float64, Matrix{Float64}}:
 2.0   ⋅ 
 1.0  1.41421

- **Проверим правильность разложения :**

In [40]:
A_reconstructed = L * L' # Восстановление матрицы A

2×2 Matrix{Float64}:
 4.0  2.0
 2.0  3.0

- **Проверка равенства определителей :**

In [41]:
det(A_reconstructed) ≈ det(A) # Проверка равенства определителей

true

#### 👉 **Преимущества Cholesky-факторизации:**

- Высокая численная устойчивость.
  
- Минимальные требования к памяти.
  
- Повышенная производительность для специальных типов матриц.

<br>

🚀 Используя **Cholesky-факторизацию** в Julia, можно эффективно и стабильно решать системы линейных уравнений, часто возникающие в прикладных задачах статистики и численного анализа.

<br>

---

<br>

### 🟢 **2.4 Сингулярное разложение (SVD, Singular Value Decomposition)**

**Сингулярное разложение (SVD)** — это факторизация любой матрицы $A$ (включая неквадратные) в виде произведения трёх матриц:

<br>

$$
A = U \cdot \Sigma \cdot V^T
$$

**где:**

- $U$ — ортогональная матрица размера $m \times m$, содержащая левые сингулярные векторы.
  
- $V$ — ортогональная матрица размера $n \times n$, содержащая правые сингулярные векторы.
  
- $\Sigma$ — прямоугольная диагональная матрица размера $m \times n$, содержащая сингулярные числа на диагонали (упорядочены по убыванию и всегда неотрицательные).
  
<br>

##### **В Julia сингулярное разложение выполняется функцией `svd()`.**

<br>

#### 🔹 **Особенности сингулярного разложения (SVD):**

- Работает для любой матрицы (квадратной и неквадратной).
  
- Диагональные элементы (сингулярные числа) матрицы $\Sigma$ неотрицательны и расположены по убыванию.
  
- Позволяет оценивать ранг матрицы и её численную устойчивость.

<br>

#### 📌 **Применение SVD:**

- **Сжатие и обработка изображений:** Уменьшение размерности данных, PCA (метод главных компонент).
  
- **Анализ данных и машинное обучение:** Рекомендательные системы, кластеризация, обработка естественного языка.
  
- **Определение ранга матрицы:** Оценка численной стабильности и качества аппроксимации данных.
  
  <br>

### 🗒  **Примеры SVD-разложения в Julia:**

#### 🔹 **Пример 1.**

1. **Определяем симметричную положительно определённую матрицу $A$:**

```julia
A = [
    25.0  15.0  -5.0;
    15.0  18.0   0.0;
    -5.0   0.0  11.0
]
```

In [42]:
A = [
    25.0  15.0  -5.0;
    15.0  18.0   0.0;
    -5.0   0.0  11.0
]

3×3 Matrix{Float64}:
 25.0  15.0  -5.0
 15.0  18.0   0.0
 -5.0   0.0  11.0

2. **Выполним сингулярное разложение при помощи функцией `svd()`:**

In [43]:
Asvd = svd(A)

SVD{Float64, Float64, Matrix{Float64}, Vector{Float64}}
U factor:
3×3 Matrix{Float64}:
 -0.783736   0.178127   0.595003
 -0.603219  -0.446485  -0.660893
  0.147937  -0.876882   0.457375
singular values:
3-element Vector{Float64}:
 37.48885385630933
 12.015683645427238
  4.495462498263424
Vt factor:
3×3 Matrix{Float64}:
 -0.783736  -0.603219   0.147937
  0.178127  -0.446485  -0.876882
  0.595003  -0.660893   0.457375

3. **Полученные компоненты факторизации извлекаются следующим образом:**
   


- Матрица $U$ (левые сингулярные векторы):

In [44]:
U = Asvd.U

3×3 Matrix{Float64}:
 -0.783736   0.178127   0.595003
 -0.603219  -0.446485  -0.660893
  0.147937  -0.876882   0.457375

- Диагональная матрица сингулярных чисел $\Sigma$:

In [45]:
S = Asvd.S

3-element Vector{Float64}:
 37.48885385630933
 12.015683645427238
  4.495462498263424

- Матрица $V$ (правые сингулярные векторы):

In [46]:
V = Asvd.V

3×3 adjoint(::Matrix{Float64}) with eltype Float64:
 -0.783736   0.178127   0.595003
 -0.603219  -0.446485  -0.660893
  0.147937  -0.876882   0.457375

- **Восстановление исходной матрицы по её SVD-разложению :**

A_reconstructed = U * Diagonal(S) * V'

### 🔹 **Пример 2 использования SVD:**

Рассмотрим новую матрицу:

```julia
A = [3 1; 1 3; 1 1]
```



In [47]:
A = [3 1; 1 3; 1 1]

3×2 Matrix{Int64}:
 3  1
 1  3
 1  1

- **Выполняем SVD-разложение :**

In [48]:
Asvd = svd(A) # Сингулярное разложение матрицы A

SVD{Float64, Float64, Matrix{Float64}, Vector{Float64}}
U factor:
3×2 Matrix{Float64}:
 -0.666667   0.707107
 -0.666667  -0.707107
 -0.333333   0.0
singular values:
2-element Vector{Float64}:
 4.242640687119286
 2.0
Vt factor:
2×2 Matrix{Float64}:
 -0.707107  -0.707107
  0.707107  -0.707107

- **Матрица $U$ (левые сингулярные векторы):**

In [49]:
U = Asvd.U # ортогональная матрица

3×2 Matrix{Float64}:
 -0.666667   0.707107
 -0.666667  -0.707107
 -0.333333   0.0

- **Диагональная матрица сингулярных чисел $\Sigma$ :**

In [50]:
S = Asvd.S # диагональная матрица

2-element Vector{Float64}:
 4.242640687119286
 2.0

- **Матрица $V$ (правые сингулярные векторы):**

In [51]:
V = Asvd.V # ортогональная матрица

2×2 adjoint(::Matrix{Float64}) with eltype Float64:
 -0.707107   0.707107
 -0.707107  -0.707107

- **Восстановление исходной матрицы по её SVD-разложению:**

In [52]:
A_reconstructed = U * Diagonal(S) * V' # Восстановление матрицы A

3×2 Matrix{Float64}:
 3.0  1.0
 1.0  3.0
 1.0  1.0

#### 👉 **Преимущества в использовании SVD-разложения:**

- Позволяет работать с любыми матрицами, включая неквадратные.
  
- Высокая численная устойчивость и надёжность.
  
- Эффективно решает задачи, связанные с обработкой больших объёмов данных, особенно при анализе данных и машинном обучении.

<br>

<br>

---

<br>

### 🟢 **2.5 Собственные разложения (Eigendecompositions -EVD)**

<br>

**Собственное разложение (Eigendecomposition)** — это представление её в виде произведения трёх матриц:

<br>

$$
A = V \Lambda V^{-1},
$$


**где:**

- $ V $ — матрица собственных векторов,
- $ \Lambda $ — диагональная матрица собственных значений.


### 🗒  **Примеры EVD-разложения в Julia:**

#### 🔹 **Пример 1.**

1. **Предположим, что у нас есть симметричная матрица $A$, созданная следующим образом :**


```julia
A = [
    25.0  15.0  -5.0;
    15.0  18.0   0.0;
    -5.0   0.0  11.0
]
```

In [53]:
A = [
    25.0  15.0  -5.0;
    15.0  18.0   0.0;
    -5.0   0.0  11.0
]

3×3 Matrix{Float64}:
 25.0  15.0  -5.0
 15.0  18.0   0.0
 -5.0   0.0  11.0

- **Или мы можем создать симметричную матрицу:**

In [54]:
Asym = A + A'# Создание симметричной матрицы

3×3 Matrix{Float64}:
  50.0  30.0  -10.0
  30.0  36.0    0.0
 -10.0   0.0   22.0

- **Чтобы вычислить собственное разложение, используем функцию `eigen`:**

In [55]:
AsymEig = eigen(Asym) # Вычисление собственных значений и векторов матрицы Asym

Eigen{Float64, Float64, Matrix{Float64}, Vector{Float64}}
values:
3-element Vector{Float64}:
  8.990924996526852
 24.03136729085444
 74.9777077126187
vectors:
3×3 Matrix{Float64}:
  0.595003  -0.178127   0.783736
 -0.660893   0.446485   0.603219
  0.457375   0.876882  -0.147937

👉 **Полученный нами результат вычисления представляет собой собственные значения и собственные векторы симметричной матрицы:**

- **Собственные значения (eigenvalues)**:
```julia
[8.990924996526852, 24.03136729085444, 74.9777077126187]
```
Собственные значения отражают, как исходная матрица масштабирует (растягивает или сжимает) пространство вдоль направлений, заданных соответствующими собственными векторами. 

Чем больше собственное значение, тем сильнее растяжение в направлении его собственного вектора.

<br>

- **Собственные векторы (eigenvectors)**:

Это столбцы (векторы) нашей матрицы –  направления (нормализованные векторы длины 1), по которым происходит масштабирование пространства. Каждый собственный вектор соответствует собственному значению с тем же индексом:

- Вектор 1 `[0.595003, -0.660893, 0.457375]` соответствует собственному значению `8.9909`.
  
- Вектор 2`[-0.178127, 0.446485, 0.876882]` соответствует собственному значению `24.0314`.
  
- Вектор 3 `[0.783736, 0.603219, -0.147937]` соответствует собственному значению `74.9777`.

<br>

>**Говоря простыми словами наша исходная матрица при умножении на данные векторы просто меняет их длину, не меняя направления.**
>**Эти три направления ортогональны (перпендикулярны друг другу), так как исходная матрица симметрична.**
>**Таким образом, полученный нами результат — не что инное как полное разложение матрицы на ортогональные направления и коэффициенты масштабирования.**

<br>

#### 🔹 Рассмотрим доступ к собственным значениям и векторам матрицы:


- **Задаем исходную матрицу A, которая будет использована для демонстрации собственных значений и векторов**

```julia
A = [
    25.0  15.0  -5.0;
    15.0  18.0   0.0;
    -5.0   0.0  11.0
]
```

A = [
    25.0  15.0  -5.0;
    15.0  18.0   0.0;
    -5.0   0.0  11.0
]

- **Создаём симметричную матрицу.**

Симметричная матрица важна для вычисления собственных значений и векторов, так как обеспечивает реальные значения и ортогональность собственных векторов.

In [56]:
Asym = A + A'

3×3 Matrix{Float64}:
  50.0  30.0  -10.0
  30.0  36.0    0.0
 -10.0   0.0   22.0

- **Выполняем EVD-разложение**

Вычисляем собственные значения и собственные векторы, получая EVD-разложение матрицы.

In [57]:
AsymEig = eigen(Asym)

Eigen{Float64, Float64, Matrix{Float64}, Vector{Float64}}
values:
3-element Vector{Float64}:
  8.990924996526852
 24.03136729085444
 74.9777077126187
vectors:
3×3 Matrix{Float64}:
  0.595003  -0.178127   0.783736
 -0.660893   0.446485   0.603219
  0.457375   0.876882  -0.147937

- **Получаем доступ к собственным значениям и векторам:**
  
  

**Собственные значения** - показывают степень масштабирования пространства по соответствующим направлениям (собственным векторам):

In [58]:
λ = AsymEig.values                     # Собственные значения
println("Собственные значения: ", λ)   # Вывод собственных значений

Собственные значения: [8.990924996526852, 24.03136729085444, 74.9777077126187]


**Собственные векторы** - задают направления, по которым происходит масштабирование:

In [59]:
V = AsymEig.vectors                    # Собственные векторы
println("Собственные векторы: \n", V)  # Вывод собственных векторов

Собственные векторы: 


[0.5950031751375643 -0.17812697674608458 0.7837359260181437; -0.660892505642328 0.446484526031047 0.6032185872473069; 0.4573754667251214 0.8768821746221844 -0.1479369266540509]


- **Проверим корректность факторизации**
  
А именно, что произведение `Asym` на `V` дает то же самое, что и произведение `V` на диагональную матрицу собственных значений

In [60]:
println("Проверка разложения: ", norm(Asym * V - V * Diagonal(λ)))  # должно быть ≈ 0

Проверка разложения: 4.8379234241467095e-14

<br>

<br>


#### 🔹 Использование EVD-факторизации для оптимизации вычислений.

**Используя EVD-факторизацию, обратная матрица вычисляется намного эффективнее, особенно для больших матриц**

In [61]:
Asym_inv_eig = inv(AsymEig)

3×3 Matrix{Float64}:
  0.0488889  -0.0407407   0.0222222
 -0.0407407   0.0617284  -0.0185185
  0.0222222  -0.0185185   0.0555556

**Проверим точность вычисления обратной матрицы :**

In [62]:
println("Эффективно вычисленная обратная матрица через EVD:\n", Asym_inv_eig)

Эффективно вычисленная обратная матрица через EVD:
[0.048888888888888864 -0.04074074074074075 0.022222222222222202; -0.04074074074074074 0.06172839506172842 -0.01851851851851851; 0.022222222222222185 -0.0185185185185185 0.055555555555555594]


**Убедимся в корректности (должна получиться единичная матрица)**

Этим шагом мы проверяем, что при умножении исходной матрицы на обратную, получается единичная матрица (I):

In [63]:
println("Проверка корректности вычисления обратной матрицы:\n", Asym_inv_eig * Asym)

Проверка корректности вычисления обратной матрицы:
[0.9999999999999988 -1.1102230246251565e-15 -2.220446049250313e-16; 8.881784197001252e-16 1.000000000000001 1.6653345369377348e-16; -1.7763568394002505e-15 -4.440892098500626e-16 1.000000000000001]


**Сравним со стандартным способом вычисления обратной матрицы, что менее эффективно. Но так ли это?**

Проверим, насколько сильно отличается результат, полученный через стандартный подход

In [64]:
Asym_inv_standard = inv(Asym)
println("Разница между двумя способами вычисления обратной матрицы: ", norm(Asym_inv_eig - Asym_inv_standard))

Разница между двумя способами вычисления обратной матрицы: 7.28583859910259e-17

#### 🔹 **Выводы:**

Использование специальных типов факторизаций в Julia позволяет:

👉 Хранить результаты вычислений компактно и логично.
  
👉 Быстро извлекать значения и векторы для дальнейших вычислений.
  
👉 Автоматически использовать оптимизированные методы, учитывающие структуру факторизации.


<br>

<br>

---

<br>

### 🟢 **2.6 Другие менее распространённые факторизации**

Помимо наиболее распространённых факторизаций, уже рассмотренных нами выше, существуют также менее известные, но важные методы разложения матриц, которые применяются в специфических и продвинутых задачах линейной алгебры и численного анализа.

<br>

### 📌 **Разложение Шура (Schur Decomposition)**

<br>

**Разложение Шура** представляет собой представление квадратной матрицы $A$ в виде произведения:

<br>

$$
A = Q \cdot T \cdot Q^*
$$

**где:**

- $Q$ — унитарная (ортогональная) матрица, то есть выполняется условие $Q^* Q = I$;
  
- $T$ — верхняя треугольная (для комплексной матрицы) или верхняя квазитреугольная матрица (для вещественной матрицы, известная как **Real Schur form**).

<br>


**Важно:**
>Однако, разработчики языка **Julia** приняли решение называть при разложении Шура унитарную матрицу в объекте не $Q$, а $Z$, чтобы избежать путаницы с другими разложениями, например **QR-разложением**, где уже используется буква $Q$. Таким образом, для ясности, это чисто техническое решение, сделанное разработчиками **Julia**.

<br>

#### 🔹 **Использование разложения Шура:**

- Удобно для вычисления собственных значений (на диагонали матрицы $T$).
  
- Применяется в теории управления и численном анализе.
  
- Эффективно используется при реализации QR-алгоритма нахождения собственных значений.

<br>

### 🗒  **Пример использования разложения Шура в Julia:**

#### 🔹 **Пример 1.**

- **Определяем матрицу $A$:**

```julia 
A = [
     4  3; 
    -5 -2
    ]
```

In [65]:
A = [4 3; -5 -2] # Создание матрицы A

2×2 Matrix{Int64}:
  4   3
 -5  -2

- **Выполняем разложение Шура (Schur decomposition) матрицы**:

In [66]:
SchurF = schur(A) # Получение матрицы Шура

Schur{Float64, Matrix{Float64}, Vector{ComplexF64}}
T factor:
2×2 Matrix{Float64}:
  1.0      0.837722
 -7.16228  1.0
Z factor:
2×2 Matrix{Float64}:
 0.811242  -0.58471
 0.58471    0.811242
eigenvalues:
2-element Vector{ComplexF64}:
 0.9999999999999999 + 2.449489742783178im
 0.9999999999999999 - 2.449489742783178im

**Получаем компоненты факторизации**

Мы помним, что разработчики языка Julia приняли решение называть унитарную матрицу в этом объекте не $Q$, а $Z$, чтобы избежать путаницы с другими разложениями.

- $Q$ – унитарная (ортогональная для вещественных матриц) матрица:

In [67]:
Q = SchurF.Z # Получение матрицы Q

2×2 Matrix{Float64}:
 0.811242  -0.58471
 0.58471    0.811242

- $𝑇$ – верхняя квазитреугольная матрица Шура (для вещественных матриц блоками 1x1 или 2x2)

In [68]:
T = SchurF.T # Получение верхнетреугольной матрицы T

2×2 Matrix{Float64}:
  1.0      0.837722
 -7.16228  1.0

👉 **Матрица $T$ имеет верхнетреугольную форму, что облегчает чтение собственных значений непосредственно с диагонали.**

<br>

---

<br>

### 📌 **Разложение Жордана (Jordan Normal Form)**

<br>

**Разложение Жордана** представляет собой представление квадратной матрицы $A$ в виде подобия с блочно-диагональной матрицей:

<br>

$$
A = P \cdot J \cdot P^{-1}
$$
**где:**

- $P$ - **Матрица перехода** - содержит в качестве столбцов собственные и обобщенные (присоединенные) векторы. Используется для перехода в новый базис, в котором исходная матрица имеет максимально простой вид (Жорданова форма).
  
- $J$ -**Матрица Жордана** - это верхнетреугольная матрица, содержащая собственные значения на диагонали и единицы над диагональю, если собственных векторов недостаточно для диагонализации.

<br>

>**Разложение применяется, когда матрица не диагонализируема, то есть ей не хватает полного набора собственных векторов.**

<br>

#### 🔹 **Использование разложения Жордана:**

- Теоретический анализ свойств матриц и операторов.
  
- Решение дифференциальных и разностных уравнений.
  
- Изучение структуры линейных операторов.

<br> 

📚 В Julia нет встроенной функции для прямого вычисления **разложения Жордана (Jordan decomposition)**, поскольку это разложение неустойчиво при численных вычислениях (чувствительно к погрешностям).

**Поэтому используют дополнительные пакеты:**
- Для численных расчетов лучше всего использовать:
   - `JordanForm.jl`
   - `LinearAlgebraX.jl`
- Для символьных расчетов рекомендуется:
   - `Symbolics.jl`
  
<br>

### 🗒  **Пример численного вычисления с помощью пакета `JordanForm.jl`**

#### 🔹 **Пример 1.**


**Установим пакет `JordanForm.jl`:**

In [69]:
# JordanForm уже указан в Project.toml этого курса, поэтому Pkg.add не нужен —
# достаточно подключить пакет. Так мы не изменяем окружение курса.
using JordanForm

- **Производим подключение пакета JordanForm для вычисления разложения Жордана**

- **Задаем исходную матрицу $A$, для которой хотим найти Жорданову форму:**

```julia
A = [6 2 1; 
     0 3 0; 
     0 0 3]
```


In [70]:
A = [6 2 1; 0 3 0; 0 0 3] # Создание матрицы A

3×3 Matrix{Int64}:
 6  2  1
 0  3  0
 0  0  3

- **Вычисляем Жорданову нормальную форму $J$ и матрицу перехода $P$:**

In [71]:
J, P = jordan_form(A) # Получение жордановой формы матрицы A

JordanFactorization{ComplexF64, ComplexF64}(ComplexF64[-0.6666666666666666 + 0.0im -0.3333333333333333 + 0.0im 1.0 + 0.0im; 1.0 + 0.0im 0.0 + 0.0im -0.0 - 0.0im; 0.0 + 0.0im 1.0 + 0.0im -0.0 - 0.0im], ComplexF64[3.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 3.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im 6.0 + 0.0im])

- **Выводим полученную Жорданову форму матрицы $A$ :**

In [72]:
println("Жорданова форма J:\n", J) # Вывод жордановой формы

Жорданова форма J:
ComplexF64

[-0.6666666666666666 + 0.0im -0.3333333333333333 + 0.0im 1.0 + 0.0im; 1.0 + 0.0im 0.0 + 0.0im -0.0 - 0.0im; 0.0 + 0.0im 1.0 + 0.0im -0.0 - 0.0im]


- **Выводим матрицу перехода (базиса), состоящую из собственных и присоединенных векторов :**

In [73]:
println("Матрица перехода P:\n", P) # Вывод матрицы перехода

Матрица перехода P:
ComplexF64

[3.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 3.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im 6.0 + 0.0im]


- **Проверяем корректность полученного разложения, вычисляя разницу между исходной матрицей и произведением матриц разложения (должна быть очень близка к 0) :**

In [74]:
println("Проверка корректности разложения: ", norm(A - P*J*inv(P))) # должно быть ≈ 0

Проверка корректности разложения: 8.552127740444998

❌ **Полученный результат означает, что разложение некорректно, так как оно должно быть очень близким к нулю (например, порядка 1e-12 или меньше).**

Это связано с тем, что алгоритмы численного вычисления Жордановой формы являются неустойчивыми и могут давать существенные ошибки из-за погрешностей округления. Именно поэтому вычисление Жордановой формы практически всегда рекомендуют выполнять:

- либо аналитически (символьно),
  
- либо для матриц, которые специально подобраны и не дают ошибок округления.

<br>

**Как же исправить эту проблему?**

:) Используйте символьное вычисление !!!

<br>

### ✅ **Символьное вычисление с пакетом `Symbolics.jl`**



**Использование символьных вычислений**

- Позволяет нам получить аналитические формулы.
  
- Удобно для теоретических и исследовательских задач, когда числовых значений элементов нет или они неизвестны заранее.
  
- Упрощает решение общих задач линейной алгебры и матричного анализа.

<br>

**Установка:**

```julia
Pkg.add("Symbolics")
Pkg.add("SymbolicUtils")
```

#### 🔹 **Пример 2.**

Корректный пример с символикой:

**Выполним установку пакета `Symbolics.js` в Julia:**

In [75]:
# Symbolics и SymbolicUtils также входят в окружение курса (Project.toml),
# поэтому их достаточно подключить.
using Symbolics, SymbolicUtils

In [76]:
using Symbolics, LinearAlgebra

**Определяем матрицу $A$:**

In [77]:
A = [6 2 1; 0 3 0; 0 0 3] # Создание матрицы A

3×3 Matrix{Int64}:
 6  2  1
 0  3  0
 0  0  3

**Производим символьное вычисление собственных значений:**

Используем функцию `eigvals(A)`

In [78]:
eigvals_A = eigvals(A) # Вычисление собственных значений матрицы A

3-element Vector{Float64}:
 3.0
 3.0
 6.0

In [79]:
println("Собственные значения:\n", eigvals_A) # Вывод собственных значений

Собственные значения:
[3.0, 3.0, 6.0]


<br>

#### 🔹 **Пример 3.**

На этом примере продемонстрируем использование символьных вычислений в Julia с помощью пакета Symbolics.jl. Убедимся, что символьные вычисления помогают аналитически исследовать матрицы, позволяя работать не только с числовыми, но и с переменными элементами матриц.

**Мы зададим символьные переменные, которые затем можно использовать в символьной матрице. Это означает, что матрица может содержать не числа, а переменные, полезные при аналитических расчётах.**

1. Задаём **символьные переменные** (элементы матрицы):

In [80]:
@variables a11 a12 a13 a21 a22 a23 a31 a32 a33 # Создание символьных переменных

9-element Vector{Num}:
 a11
 a12
 a13
 a21
 a22
 a23
 a31
 a32
 a33

 2. Определение **символьной матрицы**, которая состоит полностью из символьных элементов. Её можно использовать для вывода общих формул и аналитического исследования свойств матриц, таких как собственные значения и векторы в общем виде.

In [81]:
A = [a11 a12 a13; 
     a21 a22 a23; 
     a31 a32 a33]

3×3 Matrix{Num}:
 a11  a12  a13
 a21  a22  a23
 a31  a32  a33

3. Получаем **характеристический полином** (аналитически!):

In [82]:
λ = Symbolics.scalarize(@variables λ)[1] # Создание символьной переменной λ
char_poly = det(A - λ * I(3)) # Вычисление характеристического полинома

(a11 - λ)*((a22 - λ)*(a33 - λ) - a23*a32) - a12*(a21*(a33 - λ) - a23*a31) + a13*(a21*a32 - (a22 - λ)*a31)

In [83]:
println("Характеристический полином матрицы A:\n", char_poly) # Вывод характеристического полинома

Характеристический полином матрицы A:
(

a11 - λ)*((a22 - λ)*(a33 - λ) - a23*a32) - a12*(a21*(a33 - λ) - a23*a31) + a13*(a21*a32 - (a22 - λ)*a31)


4. Затем приводится конкретная числовая матрица, для которой будут вычислены собственные значения. Из этого видно, как можно использовать символьные методы и подходы и для конкретных числовых матриц.

In [84]:
B = [6 2 1; 
     0 3 0; 
     0 0 3] # Создание матрицы B

3×3 Matrix{Int64}:
 6  2  1
 0  3  0
 0  0  3

5. Вычисляем **характеристический полином** конкретной матрицы $B$:

In [85]:
char_poly_B = det(B - λ * I(3)) # Вычисление характеристического полинома матрицы B

(6 - λ)*((3 - λ)^2)

In [86]:
println("Характеристический полином матрицы B:\n", char_poly_B) # Вывод характеристического полинома

Характеристический полином матрицы B:
(6 - λ)*((3 - λ)^2)


**Это даст вам точный символьный вид характеристического полинома:**

$$
\det(B - λ I) = (6 - λ)(3 - λ)^2
$$


Мы можем вывести также собственные значния матрицы $B$:

In [87]:
eigvals_B = eigvals(B)  # Вычисление собственных значений матрицы B
println("Собственные значения матрицы B:\n", eigvals_B) # Вывод собственных значений матрицы B


Собственные значения матрицы B:
[3.0, 3.0, 6.0]


👉 **На этом примере мы изучили:**

- Как использовать символьную матрицу и зачем её задавать.
  
- Как получить аналитические формулы (например, характеристический полином).
  
- Как комбинировать символьный и численный подходы для более глубокого анализа матриц.

<br>

<br>


### 📚 **Дополнительные возможности с `LinearAlgebraX.jl`**

`LinearAlgebraX.jl` - это другой мощный пакет для расширенных матричных вычислений.


#### 🔹 **Пример 4.**

**Выполним установку пакета `LinearAlgebraX.jl`**:

In [88]:
# LinearAlgebraX входит в Project.toml курса.
using LinearAlgebraX

**Определяем матрицу $A$:**

In [89]:
A = [6 2 1; 0 3 0; 0 0 3] # Создание матрицы A

3×3 Matrix{Int64}:
 6  2  1
 0  3  0
 0  0  3

**Произведём вычисление Жордановой формы:**

In [90]:
J = jordan_form(A) # Получение жордановой формы матрицы A

JordanFactorization{ComplexF64, ComplexF64}(ComplexF64[-0.6666666666666666 + 0.0im -0.3333333333333333 + 0.0im 1.0 + 0.0im; 1.0 + 0.0im 0.0 + 0.0im -0.0 - 0.0im; 0.0 + 0.0im 1.0 + 0.0im -0.0 - 0.0im], ComplexF64[3.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 3.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im 6.0 + 0.0im])

In [91]:
println("Жорданова форма (J):\n", J) # Вывод жордановой формы

Жорданова форма (J):
JordanFactorization

{ComplexF64, ComplexF64}(ComplexF64[-0.6666666666666666 + 0.0im -0.3333333333333333 + 0.0im 1.0 + 0.0im; 1.0 + 0.0im 0.0 + 0.0im -0.0 - 0.0im; 0.0 + 0.0im 1.0 + 0.0im -0.0 - 0.0im], ComplexF64[

3.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 3.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im 6.0 + 0.0im])


---

<br>

#### 📝 **Заключительные выводы:**

**Факторизация** это мощный инструмент в вычислительной линейной алгебре, делающий сложные вычислительные задачи более простыми и точными. Разные типы факторизаций решают разные задачи, и выбор конкретного подхода зависит от типа исходной матрицы и задачи, которую нужно решить.


⚙️ **Как выбрать нужный тип факторизации?**

Выбор зависит от конкретной задачи и свойств матрицы:

| Свойство матрицы       | Подходящая факторизация        |
|------------------------|--------------------------------|
| Общая матрица          | LU, QR, SVD                    |
| Симметричная положительно определённая | Cholesky, LU, SVD, EVD |
| Неквадратная           | QR, SVD                        |
| Диагонализируемая      | EVD                            |

**Быстрые алгоритмы по сложности** (от простого к сложному):

- **Cholesky** (~$n^3/3$)
  
- **LU** (~$2n^3/3$)
  
- **QR** (~$2n^3$)
  
- **SVD** (~$4n^3$) — наиболее затратный, но универсальный и устойчивый.
  

**Другие менее распространённые факторизации:**

- **Разложение Шура** (Schur Decomposition).
  
- **Разложение Йордана** (Jordan Normal Form).

Эти менее распространённые факторизации широко используются в теоретических исследованиях, а также при решении специфических задач в математике, физике, теории управления и других областях, где необходимы глубокие структурные свойства матриц.

<br>

<br>

---

<br>

<br>

## 🧑‍💻 **3. Специальные матричные структуры.**

<br>

Традиционно считается, что **структура матриц** играет ключевую роль в линейной алгебре.

👉 Чтобы наглядно понять, насколько это важно, рассмотрим систему линейных уравнений с большой матрицей (например, размером 1000×1000):

In [92]:
n = 1000         # Размер матрицы
A = randn(n, n)  # Создание случайной матрицы

1000×1000 Matrix{Float64}:
 -0.809278    0.906644    1.47862    …   1.69973    -1.4315      1.84743
 -0.647818   -0.465703   -0.204725      -0.504223    0.856566    0.2826
 -1.30311    -3.02756     0.0645167     -0.501463    0.0855535   0.409802
 -0.40937     0.313274    0.871967      -1.9107     -2.06578     0.583999
  0.345855    1.63525     0.361768      -0.622608   -0.844109   -0.14249
 -1.08411     0.149609    0.0375333  …  -1.24088     1.15868    -1.21506
 -0.753845    0.0733312   1.50898       -0.477543   -0.12944    -0.0575284
  0.0524659  -0.681172   -0.667595      -3.31677     0.952902    0.549996
  0.241799    0.859097   -1.27261        0.624605    0.324436   -0.472369
  0.914617    2.51506    -0.157017       0.250049    0.687023   -0.662174
 -0.283448   -1.80643     1.31464    …  -1.94649    -0.352608    0.898392
 -0.253772   -1.33903    -1.39109       -0.28257     0.50509    -0.839484
 -0.0335347  -0.449018    0.825416      -3.2143     -0.610768   -0.459143
  ⋮            

**Julia позволяет проверять свойства матриц, автоматически используя их структуры.**

Например, проверим симметричность матрицы:

In [93]:
A_sym = A + A'  # Создание симметричной матрицы

issymmetric(A_sym)  # true

true

**Однако иногда численные погрешности могут мешать корректному определению структуры матрицы:**

In [94]:
A_sym_noisy = copy(A_sym)   # Создаем копию матрицы A_sym
A_sym_noisy[1, 2] += 5eps() # Добавляем шум к элементу матрицы

issymmetric(A_sym_noisy)    # false из-за численной погрешности

false

**К счастью, мы можем явно указать структуру матрицы, используя встроенные типы Julia, такие как:**

- `Diagonal` (диагональная),
  
- `Triangular` (треугольная),
  
- `Symmetric` (симметричная),
  
- `Hermitian` (эрмитова),
  
- `Tridiagonal` (трёхдиагональная),
  
- `SymTridiagonal` (симметричная трёхдиагональная).

Например:

In [95]:
A_sym_explicit = Symmetric(A_sym_noisy) # Явное приведение к симметричной матрице

1000×1000 Symmetric{Float64, Matrix{Float64}}:
 -1.61856    0.258826   0.175513  …   2.13507   -3.14434     1.59236
  0.258826  -0.931406  -3.23228       0.617523   0.196847    1.628
  0.175513  -3.23228    0.129033     -1.73039   -0.115642    1.22748
 -0.109338   0.393176   1.61325      -2.90046   -2.73794     0.315197
  0.268782   1.94456   -0.699411     -0.210772  -0.962586    0.988289
 -0.561258  -0.228997  -1.87302   …  -0.516715   0.615435   -0.597671
 -0.858338   1.42699   -0.223157     -0.773448  -0.753005   -1.04695
 -0.964582   0.808473  -0.475736     -1.3648     0.849099    1.24961
  0.965165   1.3668    -0.559845      0.301841   1.02612    -1.97936
  0.520454   4.58873   -0.221194     -0.834851   1.60268    -0.330864
  1.45526   -1.2238     0.160558  …  -2.25442   -0.0878868   2.32083
 -0.49339   -1.71131   -0.869291     -0.03564   -0.56136    -0.530476
  0.776289   0.267475   0.224008     -3.44708    0.605628   -0.484148
  ⋮                               ⋱                 

**Сравним скорость вычисления собственных значений для разных случаев:**

Использование типа `Symmetric` сделает вычисления примерно в 5 раз быстрее!

In [96]:
@time eigvals(A_sym);             # без явного указания структуры
@time eigvals(A_sym_noisy);       # численная погрешность
@time eigvals(A_sym_explicit);    # с явно указанной структурой

  0.365659 seconds (21 allocations: 7.988 MiB)
  2.026371 seconds (27 allocations: 7.928 MiB)


  0.539455 seconds (133.39 k allocations: 14.928 MiB, 35.54% compilation time)


<br>

#### 🚩 **Большая проблема и её решение**

Использование специальных типов, таких как `Tridiagonal` и `SymTridiagonal`, позволяет работать с очень большими трёхдиагональными матрицами, которые было бы невозможно хранить и вычислять как обычные (плотные) матрицы.

<br>

**Например, следующую задачу невозможно было бы решить на обычном ноутбуке с плотной матрицей:**

In [97]:
n = 1_000_000    # Размер матрицы
A_large = SymTridiagonal(randn(n), randn(n - 1)) # Создание большой трехдиагональной матрицы

1000000×1000000 SymTridiagonal{Float64, Vector{Float64}}:
 -1.82283    0.322569    ⋅        …    ⋅          ⋅         ⋅ 
  0.322569  -0.96513   -0.858087       ⋅          ⋅         ⋅ 
   ⋅        -0.858087   1.08148        ⋅          ⋅         ⋅ 
   ⋅          ⋅         0.516372       ⋅          ⋅         ⋅ 
   ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
   ⋅          ⋅          ⋅        …    ⋅          ⋅         ⋅ 
   ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
   ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
   ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
   ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
   ⋅          ⋅          ⋅        …    ⋅          ⋅         ⋅ 
   ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
   ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
  ⋮                               ⋱                        
   ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 


In [98]:
@time eigmax(A_large) # Вычисление максимального собственного значения

  1.614977 seconds (619.61 k allocations: 215.943 MiB, 29.91% gc time, 23.67% compilation time)


6.783770942750802

👉 **Благодаря специальным структурам матриц можно эффективно решать масштабные численные задачи, которые иначе были бы слишком ресурсоёмкими для обычных компьютеров.**

<br>

---

<br>

<br>

## ✨ **4. Общая линейная алгебра (Generic linear algebra)**

<br>

Стандартный подход к реализации численной линейной алгебры заключается в использовании подпрограмм из библиотек BLAS и LAPACK. Julia также придерживается этого подхода для матриц с элементами типов `Float32`, `Float64`, `Complex{Float32}` или `Complex{Float64}`.

Однако Julia дополнительно поддерживает **обобщённую (generic)** линейную алгебру. Это позволяет работать с матрицами и векторами, элементами которых являются, например, рациональные числа, комплексные числа произвольной точности и другие типы данных, не обязательно числовые.

<br>

### 🔢 **Рациональные числа (Rational numbers)**

Julia поддерживает рациональные числа из коробки. Рациональные числа можно создавать с помощью двойного слэша `//`:


In [99]:
r = 1//2 # Рациональное число

1//2

**Рациональные числа представляются в виде дроби, что позволяет сохранять точность вычислений, избегая ошибок округления, характерных для чисел с плавающей точкой.**

#### 🔹 **Пример: решение рациональной системы линейных уравнений**

Рассмотрим пример решения системы линейных уравнений, элементы которой — рациональные числа. Чтобы избежать проблем переполнения, характерных при работе с рациональными числами, будем использовать типы большой точности `BigInt`:

- **Создадим матрицу с рациональными элементами типа `BigInt` :**
  
  Генерация случайной рациональной матрицы 3x3

In [100]:
Arational = Matrix{Rational{BigInt}}(rand(1:10, 3, 3)) // 10

3×3 Matrix{Rational{BigInt}}:
 7//10  3//10  1//10
 7//10  7//10  9//10
 1//10  2//5   9//10

- **Создаём вектор $x$, заполненный единицами :**

In [101]:
x = fill(1, 3)

3-element Vector{Int64}:
 1
 1
 1

- **Вычисляем вектор правой части системы :**

In [102]:
b = Arational * x

3-element Vector{Rational{BigInt}}:
 11//10
 23//10
  7//5

- **Решаем систему A * solution = b :**

In [103]:
solution = Arational \ b

3-element Vector{Rational{BigInt}}:
 1
 1
 1

- **Выводим результаты :**

In [104]:
println("Матрица A:\n", Arational)
println("\nВектор b:\n", b)
println("\nРешение системы Ax = b:\n", solution)

Матрица A:
Rational{BigInt}

[7//10 3//10 1//10; 7//10 7//10 9//10; 1//10 2//5 9//10]

Вектор b:
Rational{BigInt}

[11//10, 23//10, 7//5]

Решение системы Ax = b:
Rational{BigInt}[1, 1, 1]


- **Проверка корректности :**

In [105]:
println("\nПроверка: A * solution == b? ", Arational * solution == b) # true


Проверка: A * solution == b? true

🔴 **Мы можем также явно использовать LU-факторизацию, что полезно при многократном решении системы с одной и той же матрицей :**

Наша **LU-факторизация** матрицы с рациональными элементами будет выглядеть так:

In [106]:
Arational_lu = lu(Arational)

LU{Rational{BigInt}, Matrix{Rational{BigInt}}, Vector{Int64}}
L factor:
3×3 Matrix{Rational{BigInt}}:
  1      0     0
  1      1     0
 1//7  25//28  1
U factor:
3×3 Matrix{Rational{BigInt}}:
 7//10  3//10  1//10
  0     2//5   4//5
  0      0     6//35

- **Решение с использованием факторизации :**

In [107]:
solution_lu = Arational_lu \ b

3-element Vector{Rational{BigInt}}:
 1
 1
 1

✅ **Оба способа сохраняют точность без перехода к типам с плавающей точкой.**

<br>

#### 🚩 **Преимущества обобщённой линейной алгебры:**

- Возможность работать с широким классом числовых типов, включая точные рациональные и комплексные числа произвольной точности.
  
- Исключение ошибок округления и повышение точности вычислений в теоретических и аналитических задачах.
  
- Удобство и гибкость при работе с математическими и численными задачами различной сложности.

<br>

<br>

---

<br>

## Упражнения 🚀

<br>

#### ✅ Задание 13.1 — Собственные значения

Найдите собственные значения матрицы `A` и присвойте их переменной `A_eigv`.

> 📌 **О проверке.** Результат — числа с плавающей точкой, поэтому сравнивать их
> через `==` нельзя: последний бит зависит от версии LAPACK и от процессора.
> Проверка использует `≈` (`isapprox`) — так и надо сравнивать вещественные
> результаты в любом собственном коде.

In [108]:
using LinearAlgebra

A = [140   97   74  168  131
      97  106   89  131   36
      74   89  152  144   71
     168  131  144   54  142
     131   36   71  142   36]

5×5 Matrix{Int64}:
 140   97   74  168  131
  97  106   89  131   36
  74   89  152  144   71
 168  131  144   54  142
 131   36   71  142   36

In [109]:
# Ваше решение

In [110]:
# Правильное решение

# Матрица симметрична, поэтому все её собственные значения вещественны.
# eigvals возвращает их отсортированными по возрастанию.
A_eigv = eigvals(A)

@show issymmetric(A)
@show A_eigv

# Две полезные проверки «на здравый смысл»:
@show sum(A_eigv) ≈ tr(A)     # сумма собственных значений = след
@show prod(A_eigv) ≈ det(A)   # произведение = определитель

issymmetric(A) = true


A_eigv = [-128.49322764802145, -55.88778455305688, 42.75216727931894, 87.16111477514521, 542.4677301466143]
sum(A_eigv) ≈ tr(A) = true


prod(A_eigv) ≈ det(A) = true


true

In [111]:
@assert length(A_eigv) == 5
@assert A_eigv ≈ [-128.49322764802145, -55.887784553056875,
                   42.7521672793189,   87.16111477514521,
                  542.4677301466143]
@assert sum(A_eigv) ≈ tr(A)
println("✔ Задание 13.1 выполнено")

✔ Задание 13.1 выполнено


<br>

#### ✅ Задание 13.2 — Диагональная матрица собственных значений

Постройте диагональную матрицу из собственных значений `A`
и присвойте её переменной `A_diag`.

Подумайте, какой тип выбрать: `Diagonal` хранит только диагональ,
а `Matrix` — все $n^2$ элементов, включая нули.

In [112]:
# Ваше решение

In [113]:
# Правильное решение

# Diagonal — специальный тип: хранит только диагональ и знает свою структуру,
# поэтому умножение и решение систем с ним намного дешевле.
A_diag = Diagonal(A_eigv)

display(A_diag)
@show size(A_diag)
@show typeof(A_diag)

# Сравним расход памяти с плотной матрицей того же смысла:
println("Diagonal : ", Base.summarysize(A_diag),          " байт")
println("Matrix   : ", Base.summarysize(Matrix(A_diag)),  " байт")

5×5 Diagonal{Float64, Vector{Float64}}:
 -128.493     ⋅        ⋅        ⋅         ⋅ 
     ⋅     -55.8878    ⋅        ⋅         ⋅ 
     ⋅        ⋅      42.7522    ⋅         ⋅ 
     ⋅        ⋅        ⋅      87.1611     ⋅ 
     ⋅        ⋅        ⋅        ⋅      542.468

size(A_diag) = (5, 5)


typeof(A_diag) = Diagonal{Float64, Vector{Float64}}
Diagonal : 88

 байт
Matrix   : 248 байт


**Почему это связано со спектральным разложением.**

Для симметричной матрицы верно $A = V \Lambda V^{\mathsf{T}}$, где $\Lambda$ —
как раз построенная диагональная матрица, а $V$ — ортогональная матрица
собственных векторов. Проверим:

In [114]:
F = eigen(A)
V = F.vectors
Λ = Diagonal(F.values)

@show norm(A - V * Λ * V') < 1e-9    # спектральное разложение восстанавливает A
@show norm(V' * V - I) < 1e-9        # V ортогональна: V'V = I

norm(A - V * Λ * V') < 1.0e-9 = true


norm(V' * V - I) < 1.0e-9 = true


true

In [115]:
@assert size(A_diag) == (5, 5)
@assert diag(A_diag) ≈ A_eigv
@assert A_diag ≈ Diagonal(A_eigv)
println("✔ Задание 13.2 выполнено")

✔ Задание 13.2 выполнено


<br>

#### ✅ Задание 13.3 — Нижняя треугольная матрица

Постройте нижнюю треугольную матрицу на основе `A`
(все элементы выше главной диагонали равны нулю)
и присвойте результат переменной `A_lowertri`.

In [116]:
# Ваше решение

In [117]:
# Правильное решение

# tril возвращает обычную матрицу, у которой верхний треугольник обнулён.
A_lowertri = tril(A)

display(A_lowertri)

# Не путайте с LowerTriangular: он ничего не копирует и не обнуляет,
# а лишь помечает матрицу как треугольную, чтобы Julia выбрала
# специализированный алгоритм. Значения при этом те же:
LT = LowerTriangular(A)
@show typeof(LT)
@show Matrix(LT) == A_lowertri

5×5 Matrix{Int64}:
 140    0    0    0   0
  97  106    0    0   0
  74   89  152    0   0
 168  131  144   54   0
 131   36   71  142  36

typeof(LT) = LowerTriangular{Int64, Matrix{Int64}}
Matrix(LT) == A_lowertri = 

true


true

In [118]:
@assert A_lowertri == [140    0    0    0   0
                        97  106    0    0   0
                        74   89  152    0   0
                       168  131  144   54   0
                       131   36   71  142  36]
@assert all(A_lowertri[i, j] == 0 for i in 1:5, j in 1:5 if j > i)
println("✔ Задание 13.3 выполнено")

✔ Задание 13.3 выполнено


<br>

#### ✅ Задание 13.4 — Выбор разложения

Для каждой матрицы ниже определите, какое разложение уместно, и запишите
ответы в вектор строк `choices` — по одному из значений
`"cholesky"`, `"lu"`, `"qr"`.

| | Матрица | Свойства |
|---|---|---|
| 1 | `M1` | квадратная, симметричная, положительно определённая |
| 2 | `M2` | квадратная, несимметричная |
| 3 | `M3` | прямоугольная 5×2 (переопределённая система) |

Проверяйте свойства **кодом** (`issymmetric`, `isposdef`, `size`),
а не на глаз.

In [119]:
M1 = [ 4.0  1.0  0.0
       1.0  3.0  1.0
       0.0  1.0  2.0 ]

M2 = [ 1.0  2.0  3.0
       0.0  1.0  4.0
       5.0  6.0  0.0 ]

M3 = randn(5, 2)

5×2 Matrix{Float64}:
  0.817325   0.185164
  2.31807    0.906234
  0.604017  -0.515798
 -0.123468  -1.98957
  1.13215    0.209552

In [120]:
# Ваше решение

In [121]:
# Правильное решение

function choose(M)
    if size(M, 1) != size(M, 2)
        return "qr"                                  # прямоугольная → наименьшие квадраты
    elseif issymmetric(M) && isposdef(M)
        return "cholesky"                            # СПО → самый дешёвый вариант
    else
        return "lu"                                  # общий квадратный случай
    end
end

choices = [choose(M1), choose(M2), choose(M3)]

for (name, M, c) in zip(("M1", "M2", "M3"), (M1, M2, M3), choices)
    println(rpad(name, 4), " size=", size(M),
            "  symmetric=", issymmetric(M),
            "  posdef=", size(M,1) == size(M,2) && issymmetric(M) && isposdef(M),
            "  → ", c)
end

M1   size=

(3, 3)  symmetric=true  posdef=true  → cholesky
M2  

 size=(3, 3)  symmetric=false  posdef=false  → lu
M3   size=(5, 2)  symmetric=false  posdef=false  → qr


In [122]:
@assert choices == ["cholesky", "lu", "qr"]
println("✔ Задание 13.4 выполнено")

✔ Задание 13.4 выполнено


<br>

#### ✅ Задание 13.5 — Переиспользование факторизации

Дана матрица `K` и три правые части `r1`, `r2`, `r3`.

Разложите `K` **один раз**, сохраните разложение в `Kf`, а затем решите
все три системы, собрав решения в вектор `sols`.

В `max_resid` запишите максимальную из трёх норм невязки.

In [123]:
K = [ 6.0  2.0  1.0
      2.0  5.0  2.0
      1.0  2.0  4.0 ]

r1 = [ 9.0,  9.0,  7.0]
r2 = [ 1.0,  0.0,  0.0]
r3 = [-3.0,  4.0,  2.0]

3-element Vector{Float64}:
 -3.0
  4.0
  2.0

In [124]:
# Ваше решение

In [125]:
# Правильное решение

# K симметрична и положительно определена, поэтому разложение Холецкого —
# самый дешёвый корректный выбор. Проверяем это, а не предполагаем.
@show issymmetric(K), isposdef(K)

Kf = cholesky(K)                        # раскладываем ОДИН раз

sols = [Kf \ r for r in (r1, r2, r3)]   # три решения по цене подстановок

max_resid = maximum(norm(K * x - r) for (x, r) in zip(sols, (r1, r2, r3)))

for (i, x) in enumerate(sols)
    println("x$i = ", round.(x, digits = 6))
end
@show max_resid

(issymmetric(K), isposdef(K)) = (true, true)


x1 = 

[1.0, 1.0, 1.0]
x2 = [0.192771, -0.072289, -0.012048]
x3 = [-0.891566, 1.084337, 0.180723]
max_resid = 1.9860273225978185e-15


1.9860273225978185e-15

In [126]:
@assert length(sols) == 3
@assert max_resid < 1e-10
@assert K * sols[1] ≈ r1
@assert sols[2] ≈ K \ r2      # тот же ответ, что и прямым решением
println("✔ Задание 13.5 выполнено")

✔ Задание 13.5 выполнено


<br>

---

<br>

## 📝 Итоги занятия

| Разложение | Для каких матриц | Типичное применение |
|---|---|---|
| `lu` | любая квадратная | решение систем, определитель |
| `cholesky` | симметричная **положительно определённая** | вдвое дешевле LU |
| `qr` | любая, в том числе прямоугольная | наименьшие квадраты, устойчивость |
| `eigen` | квадратная | собственные значения и векторы |
| `svd` | любая | ранг, псевдообратная, сжатие данных |
| `schur` | квадратная | теория, устойчивые вычисления |

**Три вещи, которые стоит унести:**

1. **Проверяйте предусловия.** `isposdef` перед `cholesky` — не формальность:
   при нарушении будет `PosDefException`.
2. **Раскладывайте один раз.** Если правых частей много, сохраните факторизацию:
   $O(n^3)$ платится однажды, дальше только $O(n^2)$.
3. **Сообщайте Julia о структуре.** `Symmetric`, `Diagonal`, `SymTridiagonal`
   включают специализированные алгоритмы — но это обещание, которое Julia
   не проверяет.

<br>

### 🎓 Курс завершён

Вы прошли путь от `println("Hello")` до спектральных разложений.
Дальше стоит посмотреть в сторону `DifferentialEquations.jl`,
`DataFrames.jl`, `Flux.jl` или `JuMP.jl` — в зависимости от ваших задач.

---

[← Двенадцатое занятие. Линейная алгебра в Julia](12%20-%20Linear%20algebra%20in%20Julia.ipynb) | [Оглавление](README.md) | *(конец курса)*